In [ ]:
import sys
from pathlib import Path
import re
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode
from bs4 import BeautifulSoup

_PROJECT_ROOT = Path.cwd().resolve()
if _PROJECT_ROOT.name == "notebooks":
    _PROJECT_ROOT = _PROJECT_ROOT.parent.parent
elif _PROJECT_ROOT.name == "podscan":
    _PROJECT_ROOT = _PROJECT_ROOT.parent
sys.path.insert(0, str(_PROJECT_ROOT))

from google_utils.google_sheet import GoogleSheetService
from data.constants import CREDENTIALS_FILE

In [ ]:
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1pPF2clctk6NxAfI5AjgUJhasy8YrT1b_9XEf6MGbJe0/edit?gid=1458675702#gid=1458675702"
# ── Configurable constants ──────────────────────────────────────────────────
PODSCAN_PROFILE_PATH = "/Users/utkarshumang/.crawl4ai/profiles/podcast_profile"
PAGE_LOAD_WAIT = 10        # seconds to wait after page loads before capturing HTML
BETWEEN_URL_DELAY = 10.0    # mean delay (seconds) between starting each URL crawl
CONCURRENCY = 2            # max parallel crawls
ENTITY_ID_COL_LETTER = "X" # column letter to write entity_id into

ENTITY_COL_INDEX = ord(ENTITY_ID_COL_LETTER) - ord("A")  # 23 for "X"

In [ ]:
sheet_service = GoogleSheetService(credentials_file=CREDENTIALS_FILE)
spreadsheet_id = sheet_service.extract_spreadsheet_id(SPREADSHEET_URL)
success, result = sheet_service.list_sheets(spreadsheet_id)
if success:
    print("Available sheets:", result)
else:
    print("Error:", result)

In [ ]:
# ## The Filtering and Cleaning Scipt 

# def filter_unique_episodes(data: list, episode_id_col: str, status_col: str) -> list:
#     """
#     Keep unique episode_id; when duplicates exist with Pending and Completed, prefer Completed.
#     """
#     if not data:
#         return []
#     headers = data[0]
#     try:
#         episode_id_idx = headers.index(episode_id_col)
#         status_idx = headers.index(status_col)
#     except ValueError:
#         return data  # columns not found, return as-is

#     result_map = {}
#     for row in data[1:]:
#         episode_id = row[episode_id_idx] if episode_id_idx < len(row) else ""
#         if not episode_id or not str(episode_id).strip():
#             continue
#         status = row[status_idx] if status_idx < len(row) else ""
#         if episode_id not in result_map:
#             result_map[episode_id] = row
#         else:
#             existing_status = result_map[episode_id][status_idx] if status_idx < len(result_map[episode_id]) else ""
#             if existing_status == "Pending" and status == "Completed":
#                 result_map[episode_id] = row

#     return [headers] + list(result_map.values())


# # Main loop: filter duplicate episode_ids across all sheets except List Info
# success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
# if not success or not isinstance(sheet_names, list):
#     print(f"Cannot list sheets: {sheet_names}")
# else:
#     sheets_to_process = [s for s in sheet_names if s != "List Info"]
#     for sheet_name in sheets_to_process:
#         success, data = sheet_service.get_sheet_values(spreadsheet_id, f"{sheet_name}!A:Z")
#         if not success:
#             print(f"[{sheet_name}] Error: {data}")
#             continue
#         if not data:
#             print(f"[{sheet_name}] No data, skipping")
#             continue

#         headers = data[0]
#         if "episode_id" not in headers or "analysis_status" not in headers:
#             print(f"[{sheet_name}] Missing episode_id or analysis_status columns, skipping")
#             continue

#         rows_before = len(data) - 1
#         filtered = filter_unique_episodes(data, "episode_id", "analysis_status")
#         rows_after = len(filtered) - 1

#         success, msg = sheet_service.clear_and_rewrite_sheet(spreadsheet_id, sheet_name, filtered)
#         if success:
#             print(f"[{sheet_name}] {rows_before} -> {rows_after} rows ({rows_before - rows_after} duplicates removed)")
#         else:
#             print(f"[{sheet_name}] Error: {msg}")

In [ ]:
# ── Browser config (reuses logged-in Podscan profile) ──────────────────────
browser_config = BrowserConfig(
    headless=True,
    use_managed_browser=True,
    user_data_dir=PODSCAN_PROFILE_PATH,
    browser_type="chromium",
    accept_downloads=True,
    downloads_path="/tmp/crawl4ai_downloads",
)

# ── Crawl config (timing & parallelism) ────────────────────────────────────
crawl_config = CrawlerRunConfig(
    delay_before_return_html=PAGE_LOAD_WAIT,
    mean_delay=BETWEEN_URL_DELAY,
    max_range=0.5,
    semaphore_count=CONCURRENCY,
    cache_mode=CacheMode.BYPASS,
    wait_until="commit",
    page_timeout=120000,
    verbose=True,
)

# ── Helpers ─────────────────────────────────────────────────────────────────
PODSCAN_EPISODE_URL = "https://podscan.fm/dashboard/podcasts/{podcast_id}/episode/{episode_id}"
ENTITY_ID_PATTERN = re.compile(r"(en_[a-zA-Z0-9]+)")

def build_episode_url(podcast_id: str, episode_id: str) -> str:
    return PODSCAN_EPISODE_URL.format(podcast_id=podcast_id, episode_id=episode_id)

def extract_first_entity_id(html: str) -> str:
    """Extract entity_id from the Guests section (not Hosts) using CSS selector."""
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    guests_dt = soup.find("dt", string=lambda t: t and "Guests" in t)
    if not guests_dt:
        return ""
    guests_dd = guests_dt.find_next_sibling("dd")
    if not guests_dd:
        return ""
    link = guests_dd.select_one('a[href*="/dashboard/entities/en_"]')
    if not link:
        return ""
    match = ENTITY_ID_PATTERN.search(link.get("href", ""))
    return match.group(1) if match else ""

print("Config ready.")
print(f"  Profile  : {PODSCAN_PROFILE_PATH}")
print(f"  Page wait: {PAGE_LOAD_WAIT}s | Between URLs: {BETWEEN_URL_DELAY}s | Concurrency: {CONCURRENCY}")

In [ ]:
import asyncio
import time

async def process_sheets():
    success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
    if not success:
        print(f"Cannot list sheets: {sheet_names}")
        return

    sheets_to_process = [s for s in sheet_names if s != "List Info"]
    print(f"Sheets to process: {len(sheets_to_process)}\n")

    async with AsyncWebCrawler(config=browser_config) as crawler:
        for sheet_name in sheets_to_process:
            print(f"{'='*60}")
            print(f"Sheet: {sheet_name}")
            print(f"{'='*60}")

            try:
                success, data = sheet_service.get_sheet_values(
                    spreadsheet_id, f"'{sheet_name}'!A:Z"
                )
                if not success or not data:
                    print(f"  No data or error, skipping.\n")
                    continue

                headers = data[0]
                try:
                    podcast_id_idx = headers.index("podcast_id")
                    episode_id_idx = headers.index("episode_id")
                except ValueError as e:
                    print(f"  Missing column: {e}, skipping.\n")
                    continue

                rows_to_process = []
                skipped_existing = 0
                skipped_empty = 0

                for i, row in enumerate(data[1:], start=2):
                    podcast_id = row[podcast_id_idx] if podcast_id_idx < len(row) else ""
                    episode_id = row[episode_id_idx] if episode_id_idx < len(row) else ""
                    existing_entity = row[ENTITY_COL_INDEX] if ENTITY_COL_INDEX < len(row) else ""

                    if existing_entity.strip():
                        skipped_existing += 1
                        continue
                    if not podcast_id.strip() or not episode_id.strip():
                        skipped_empty += 1
                        continue

                    url = build_episode_url(podcast_id.strip(), episode_id.strip())
                    rows_to_process.append((i, url))

                total_rows = len(data) - 1
                print(f"  Total rows: {total_rows} | To process: {len(rows_to_process)} | "
                      f"Skipped (existing): {skipped_existing} | Skipped (empty): {skipped_empty}")

                if not rows_to_process:
                    print(f"  Nothing to do.\n")
                    continue

                updates = []
                found_count = 0
                failed_count = 0

                for idx, (row_idx, url) in enumerate(rows_to_process, 1):
                    print(f"  [{idx}/{len(rows_to_process)}] row {row_idx}: {url}")
                    entity_id = ""

                    try:
                        result = await crawler.arun(url, config=crawl_config)
                        if result.success and result.html:
                            entity_id = extract_first_entity_id(result.html)
                            if entity_id:
                                print(f"    -> entity: {entity_id}")
                            else:
                                print(f"    -> no entity found in HTML ({len(result.html)} chars)")
                        else:
                            print(f"    -> crawl returned success={result.success}, html={'yes' if result.html else 'empty'}")
                    except Exception as e:
                        failed_count += 1
                        err_msg = str(e)[:120]
                        print(f"    -> ERROR: {err_msg}")

                    if entity_id:
                        found_count += 1
                        updates.append({
                            "range": f"'{sheet_name}'!{ENTITY_ID_COL_LETTER}{row_idx}",
                            "values": [[entity_id]],
                        })

                    if idx < len(rows_to_process):
                        await asyncio.sleep(BETWEEN_URL_DELAY)

                print(f"\n  Summary: found={found_count} | failed={failed_count} | "
                      f"no_entity={len(rows_to_process) - found_count - failed_count}")

                if updates:
                    sheet_service.service.spreadsheets().values().batchUpdate(
                        spreadsheetId=spreadsheet_id,
                        body={"valueInputOption": "RAW", "data": updates},
                    ).execute()
                    print(f"  Wrote {len(updates)} entity IDs to col {ENTITY_ID_COL_LETTER}.")

            except Exception as e:
                print(f"  ERROR on sheet '{sheet_name}': {e}")

            print()

    print("All sheets processed.")

await process_sheets()